# Pokémon Analayis

Quick analysis of the Pokémon dataset here: https://www.kaggle.com/datasets/rounakbanik/pokemon

### Install requirements

In [ ]:
%pip install scikit-learn pandas matplotlib plotly nbformat

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

### Import and Define Columns

In [ ]:
# Read csv
df = pd.read_csv('pokemon.csv')

# List columns
print(df.columns.tolist())

In [ ]:
# Define column groups for important stats to understand dataset

# Generations
col_gen = ['generation']

# Identifiers
col_id = ['name', 'pokedex_number', 'classfication']

# Base stats
col_base_stats = ['name', 'hp', 'attack', 'defense', 'sp_attack', 'sp_defense', 'speed']

# Two types
col_types = ['type1', 'type2']

# Type multiplier matchup columns - source pokemon to that type
col_matchup = [
     'against_bug', 'against_dark', 'against_dragon', 'against_electric',
     'against_fairy', 'against_fight', 'against_fire', 'against_flying',
     'against_ghost', 'against_grass', 'against_ground', 'against_ice',
     'against_normal', 'against_poison', 'against_psychic', 'against_rock',
     'against_steel', 'against_water'
]

# Spawn
col_spawn = ['capture_rate', 'percentage_male', 'base_egg_steps']

### Overall Stats

In [ ]:
# Number of Pokemon per generation in this dataset

# Pull counts per generation, this does not cover overlap in generations
counts = df['generation'].value_counts().tolist()

# Generation List
generations = df['generation'].drop_duplicates().tolist()

# Bar Chart
plt.bar(generations, counts)
plt.title("Unique Pokemon by Generation")
plt.xlabel("Generation")
plt.ylabel("Pokemon count")
plt.plot()

### Type Distribution

In [ ]:
# Pokemon type count overall

# Map type counts - stack primary type1 under type2

# Type 1 list + count
type_count1 = df['type1'].value_counts().tolist()
types = df['type1'].drop_duplicates().tolist()

# Type 2 list + count
type_count2 = df['type2'].value_counts().tolist()

# Change width
plt.figure(figsize=(15,5))

# Stack for type1 - bottom
plt.bar(types, type_count1)

# Stack for type 2 - top
plt.bar(types, type_count2, bottom=type_count1, color='r')

# Plot
plt.title("Count by Types")
plt.xlabel("Type")
plt.ylabel("Count")
plt.plot()

### Eeveelution Comparisons

In [ ]:
# Evee count by generation

# Define list of Eeveelutions
eeveelutions = ['Vaporeon', 'Jolteon', 'Flareon', 'Espeon', 'Umbreon', 'Leafeon', 'Glaceon', 'Sylveon']

# Filter df on evee names
evee_df = df[df['name'].isin(eeveelutions)]

# Count evee count by generation
evee_count = evee_df['generation'].value_counts().sort_index()

# Bar Chart
plt.bar(evee_count.index, evee_count.values)
plt.title("Eeveelutions by Generation")
plt.xlabel("Generation")
plt.ylabel("Eeveelutions count")
plt.plot()

In [ ]:
# Combined radar chart for overall stats for each evolution

# Prep df
radar_df = df.loc[df['name'].isin(eeveelutions), col_base_stats]

# Define figure
fig = go.Figure()

# Loop over stats to generate traces
for evee in eeveelutions:

     # Pull stats for that evee
     stats = radar_df.loc[radar_df['name'] == evee].values.flatten()

     # Add trace per pokemon
     fig.add_trace(go.Scatterpolar(
          r=stats,
          theta=col_base_stats,
          fill='toself',
          name=evee
     ))

# Show figure
fig.show()

In [ ]:
# Example radar chart for speed
fig = px.line_polar(radar_df, r='speed', theta='name', line_close=True)

# Draw figure for speed as example
fig.update_traces(fill='toself')
fig.show()

In [ ]:
# Best evee per stat
for stat in col_base_stats:
    print(f'Stat: {stat} - {evee_df.at[evee_df[stat].idxmax(), "name"]}')

In [ ]:
# Generate radar charts for each stat, traces for each Pokémon on each

# Define column width
width_array = [0.80] * 4

# Define figure with 2x4
fig = make_subplots(
     rows=2,
     cols=4,
     column_widths=width_array,
     specs=[
          [{'type': 'polar'}] * 4,
          [{'type': 'polar'}] * 4]
     )

# Iterate over stats and generate 1 radar per stat
for i, stat in enumerate(col_base_stats):

     # Define row/column for plot
     row = (i // 4) + 1
     col = (i % 4) + 1

     # Define subplot/trace for each stat
     fig.add_trace(
          go.Scatterpolar(
               r=radar_df[stat],
               theta=radar_df['name'],
               fill='toself',
               name=stat
          ),
          row = row,
          col= col
     )

# Plot
fig.show()